# 03 — BERT fine-tuning (Google Colab, GPU)

**This notebook is meant to run on a GPU — Google Colab or a local CUDA card.**

## Before you start — 3 steps

1. Upload this notebook: [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook**
2. Switch on the GPU: **Runtime → Change runtime type → Hardware accelerator: T4 GPU** → Save
3. Upload the three parquet files from the `data/processed/` folder (folder icon on the left → upload),
   OR mount Google Drive with the cell below.
4. *(Recommended)* Also upload the project's `src/` folder next to the notebook. Then every
   setting comes from `src/config.py` and the class weights from `src/evaluation.py`, exactly
   as in the other notebooks. Without it the notebook still runs, using the fallback values
   in the settings cell.

> **Why not on the laptop CPU?** BERT-base has 110 million parameters. Fine-tuning
> it on 50,000 reviews for 3 epochs takes **8–15 hours** on a CPU and
> **15–25 minutes** on a free Colab T4 GPU. Same code, 30× faster.

> **Colab time limits:** the free tier disconnects after ~90 minutes of
> inactivity and after at most ~12 hours of running. That is why checkpoints
> are saved to Drive, and why we do not run more than 3 epochs.

## 1. Checking the environment

In [ ]:
# Is it running on a GPU at all? If you see "CPU" here, switch it:
# Runtime -> Change runtime type -> T4 GPU
import torch

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("!!! NO GPU — switch the runtime type, otherwise it will run for hours !!!")

In [ ]:
# Only needed on Colab. Skip it when running locally.
!pip install -q transformers datasets accelerate evaluate
print("Installation done.")

In [ ]:
import os, sys, time, json
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset

# --- Settings: from the project's config.py whenever it is reachable ---
# Locally (from the notebooks/ folder) or on Colab with the src/ folder uploaded
# next to the notebook, every value comes from src/config.py — the single source
# of truth, the same as in notebooks 01 and 02.
# On a bare Colab runtime without src/ the fallback values below are used.
# They are COPIES of config.py: if you change config.py, update them too.
for p in ("..", "."):
    if p not in sys.path:
        sys.path.append(p)

try:
    from src.config import CFG
    from src.evaluation import class_weights as shared_class_weights
    SEED          = CFG.random_seed
    MODEL_NAME    = CFG.model_name
    MAX_LENGTH    = CFG.max_length
    LEARNING_RATE = CFG.learning_rate
    NUM_EPOCHS    = CFG.num_epochs
    WEIGHT_DECAY  = CFG.weight_decay
    WARMUP_RATIO  = CFG.warmup_ratio
    LABELS        = list(CFG.labels)
    BATCH_LARGE_GPU                 = CFG.batch_size        # e.g. Colab T4, 16 GB
    BATCH_SMALL_GPU, ACCUM_SMALL_GPU = CFG.local_batch_size, CFG.local_grad_accum
    CONFIG_SOURCE = "src/config.py"
except ImportError:
    shared_class_weights = None
    SEED          = 42
    MODEL_NAME    = "bert-base-uncased"
    MAX_LENGTH    = 256
    LEARNING_RATE = 2e-5
    NUM_EPOCHS    = 3
    WEIGHT_DECAY  = 0.01
    WARMUP_RATIO  = 0.1
    LABELS        = ["negative", "neutral", "positive"]
    BATCH_LARGE_GPU                 = 32
    BATCH_SMALL_GPU, ACCUM_SMALL_GPU = 4, 8
    CONFIG_SOURCE = "fallback values in this cell (src/ not found)"

LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

# --- Reproducibility ---
import random
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# --- Batch size from the GPU memory ---
# A 16 GB card (Colab T4) takes the full batch of 32 at once. A 4 GB card
# (e.g. RTX 3050 Ti) fits only 4 reviews of 256 tokens, so it accumulates the
# gradient over 8 steps: 4 x 8 = the same EFFECTIVE batch of 32, so the result
# is mathematically the same, it is just computed in more steps.
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9
           if torch.cuda.is_available() else 0.0)
if vram_gb >= 12:
    BATCH_SIZE, GRAD_ACCUM = BATCH_LARGE_GPU, 1
else:
    BATCH_SIZE, GRAD_ACCUM = BATCH_SMALL_GPU, ACCUM_SMALL_GPU

print(f"Settings from: {CONFIG_SOURCE}")
print(f"Model: {MODEL_NAME} | max_len {MAX_LENGTH} | batch {BATCH_SIZE} x accum {GRAD_ACCUM} "
      f"= {BATCH_SIZE * GRAD_ACCUM} effective | lr {LEARNING_RATE} | VRAM {vram_gb:.1f} GB")

## 2. Loading the data

**Option A:** manual upload — drag the three parquet files into the file browser on the left.
**Option B:** mount Google Drive (recommended, because it survives a restart).

In [ ]:
# --- Option B: Google Drive ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/thesis/data/processed"

# --- Option A: manual upload (default) ---
DATA_DIR = "/content"

train_df = pd.read_parquet(f"{DATA_DIR}/booking_train.parquet")
val_df   = pd.read_parquet(f"{DATA_DIR}/booking_val.parquet")
test_df  = pd.read_parquet(f"{DATA_DIR}/booking_test.parquet")

print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")
print(train_df["sentiment"].value_counts().to_string())

In [ ]:
# IMPORTANT: the 'text_bert' column is used, NOT 'text_clean'!
#
# Why? BERT learns from context. If you remove the stopwords and the
# negations, "not clean" becomes "clean", and you teach exactly the opposite.
# text_bert has only had the real noise (HTML, URLs) cleaned off.
#
# DESCRIBE this decision in chapter 6.1 of the thesis — it is a concrete,
# justified methodological difference between the two model families.
TEXT_COL = "text_bert"

def to_hf_dataset(df):
    d = pd.DataFrame({
        "text": df[TEXT_COL].astype(str),
        "label": df["sentiment"].astype(str).map(LABEL2ID),
    })
    return Dataset.from_pandas(d, preserve_index=False)

ds_train = to_hf_dataset(train_df)
ds_val   = to_hf_dataset(val_df)
ds_test  = to_hf_dataset(test_df)
ds_train[0]

## 3. Tokenisation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    # NO padding here: the DataCollator pads each batch to its longest sample.
    # This way no computation is wasted on unnecessary [PAD] tokens -> faster.
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds_train = ds_train.map(tokenize, batched=True, remove_columns=["text"])
ds_val   = ds_val.map(tokenize,   batched=True, remove_columns=["text"])
ds_test  = ds_test.map(tokenize,  batched=True, remove_columns=["text"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(ds_train)

In [ ]:
# How many reviews hit the MAX_LENGTH limit? The text ends with the NEGATIVE
# part, so truncation cuts negative content first (see config.py). If more
# than a few percent are truncated, raise MAX_LENGTH in config.py — but know
# that the running time grows roughly quadratically with the length.
lens = [len(x) for x in ds_train["input_ids"]]
print(f"Token length — mean {np.mean(lens):.0f}, median {np.median(lens):.0f}, "
      f"95%: {np.percentile(lens, 95):.0f}, max {max(lens)}")
print(f"{np.mean(np.array(lens) >= MAX_LENGTH):.1%} of the samples reach the "
      f"{MAX_LENGTH}-token limit (they will be truncated).")

## 4. Model and metrics

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
)

n_total = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {n_total/1e6:.1f} million")
print("(For comparison: your TF-IDF + LogReg model has about 150 thousand weights.)")

In [ ]:
def compute_metrics(eval_pred):
    """The SAME metrics as for the classical models — without this the RQ1
    comparison would not be fair."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0)
    _, _, f_wt, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": p_mac,
        "recall_macro": r_mac,
        "f1_macro": f_mac,
        "f1_weighted": f_wt,
    }

## 5. Class weights — handling the imbalance

The data is **not balanced**: about 13% negative, 22% neutral, 64% positive.
By default BERT's loss function penalises every error equally — so the fastest
way to reduce the loss is for the model to simply predict the majority.

The solution is the same as for the classical models (`class_weight='balanced'`),
only here it has to be passed to the loss function **manually**. The weights are
computed by `class_weights()` in `src/evaluation.py`, the same function the rest
of the project uses.

> **What we do NOT do:** we do not resample the data. The validation and test
> sets must reflect the REAL distribution, otherwise the measured performance
> does not estimate the real-world performance.

In [ ]:
import torch.nn as nn

counts = train_df["sentiment"].astype(str).value_counts().reindex(LABELS)
n = len(train_df)

if shared_class_weights is not None:
    # The shared implementation: src/evaluation.py -> class_weights()
    w = shared_class_weights(train_df["sentiment"], LABELS)
else:
    # Bare Colab without src/: the same formula as src/evaluation.class_weights
    #   weight_c = n_total / (n_classes * n_c)
    w = {l: n / (len(LABELS) * counts[l]) for l in LABELS}
weights = np.array([w[l] for l in LABELS])

print("Class distribution and weights on the training set:")
for lbl, c, wt in zip(LABELS, counts, weights):
    print(f"  {lbl:9s}  n={c:7,}  ({c/n:5.1%})  ->  weight {wt:.3f}")

class_weights_t = torch.tensor(weights, dtype=torch.float32)
print(f"\nErrors on the negative class get {weights[0]/weights[2]:.1f}x the weight")
print("of the positive class — so the model cannot settle for predicting the majority.")

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer with a weighted loss function.

    Only compute_loss is overridden; everything else is the Trainer default.
    **kwargs is needed because newer transformers versions also pass a
    `num_items_in_batch` parameter."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_t.to(logits.device))
        loss = loss_fct(logits.view(-1, len(LABELS)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

print("WeightedTrainer ready.")

## 6. Training

In [ ]:
args = TrainingArguments(
    output_dir="./bert_sentiment",
    seed=SEED,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=BATCH_SIZE * 4,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,   # gradual warm-up of the learning rate

    # fp16: half-precision computation. ~2x speed-up on a T4 GPU with a barely
    # measurable loss of precision. Switch it off if you run on a CPU.
    fp16=torch.cuda.is_available(),

    eval_strategy="epoch",     # older transformers: evaluation_strategy
    save_strategy="epoch",
    logging_steps=100,

    # Keep the best model, not the last one — and based on MACRO-F1, because
    # that is the relevant metric in the imbalanced 3-class task.
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,        # do not fill up the Colab disk

    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print(f"Estimated optimiser steps: {len(ds_train) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS:,}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# ===== TRAINING STARTS HERE =====
# 50,000 samples, T4 GPU, 3 epochs: about 15-25 minutes.
# Do NOT close the browser tab and do not leave it alone for 90+ minutes!
t0 = time.time()
train_result = trainer.train()
train_minutes = (time.time() - t0) / 60

print(f"\nTraining time: {train_minutes:.1f} minutes")
print("WRITE THIS NUMBER DOWN — it is needed for the trade-off analysis of the thesis "
      "(the classical models trained in seconds).")

In [ ]:
# Learning curve — a good figure for the thesis, and it shows whether there is overfitting.
import matplotlib.pyplot as plt

hist = pd.DataFrame(trainer.state.log_history)
tr = hist.dropna(subset=["loss"])
ev_ = hist.dropna(subset=["eval_loss"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tr["epoch"], tr["loss"], label="train loss", color="#4a7ba7")
axes[0].plot(ev_["epoch"], ev_["eval_loss"], "o-", label="val loss", color="#c1483b")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Learning curve")
axes[0].legend(frameon=False)

axes[1].plot(ev_["epoch"], ev_["eval_f1_macro"], "o-", color="#3d7a5a", label="macro-F1")
axes[1].plot(ev_["epoch"], ev_["eval_accuracy"], "s--", color="#8a8680", label="accuracy")
axes[1].set(xlabel="Epoch", ylabel="Value", title="Validation performance")
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig("03_bert_training_curve.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nIf the val loss starts rising while the train loss falls -> OVERFITTING.")
print("Then fewer epochs or a larger weight_decay is needed. Describe it in the thesis!")

## 7. Evaluation on the test set

In [ ]:
test_out = trainer.predict(ds_test)
test_metrics = {k.replace("test_", ""): v for k, v in test_out.metrics.items()
                if k.startswith("test_") and isinstance(v, float)}

print("=== BERT — TEST results ===")
for k, v in test_metrics.items():
    print(f"  {k:22s} {v:.4f}")

y_pred = np.argmax(test_out.predictions, axis=-1)
y_true = np.array(ds_test["label"])

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=LABELS, digits=3, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cmn = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.imshow(cmn, cmap="Blues")
ax.set_xticks(range(3), LABELS, rotation=20); ax.set_yticks(range(3), LABELS)
ax.set(xlabel="Predicted", ylabel="True", title="BERT — confusion matrix")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cmn[i,j]:.1%}\n(n={cm[i,j]})", ha="center", va="center",
                fontsize=9, color="white" if cmn[i,j] > cmn.max()/2 else "black")
plt.tight_layout()
plt.savefig("03_bert_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. RQ1 — The big comparison

In [ ]:
# Enter the results of notebook 02 here (results/tables/02_classical_validation_results.csv)
CLASSICAL = {
    "Multinomial NB":      {"accuracy": 0.000, "f1_macro": 0.000, "f1_weighted": 0.000},
    "Logistic Regression": {"accuracy": 0.000, "f1_macro": 0.000, "f1_weighted": 0.000},
    "Linear SVM":          {"accuracy": 0.000, "f1_macro": 0.000, "f1_weighted": 0.000},
}
CLASSICAL["BERT-base"] = {k: test_metrics.get(k, np.nan)
                          for k in ["accuracy", "f1_macro", "f1_weighted"]}

comp = pd.DataFrame(CLASSICAL).T.round(4)
comp["train_time_min"] = [0.1, 0.5, 0.3, round(train_minutes, 1)]
display(comp)
comp.to_csv("03_rq1_model_comparison.csv")

print("\nThis table is the MAIN RESULT of the thesis (RQ1).")
print("What has to be written with it: not only WHICH one is better, but BY HOW MUCH")
print("and AT WHAT COST. If BERT brings +5 points for 100x the compute, that is a")
print("real engineering decision that the IT manager of a hotel chain would also make.")

## 9. Saving the model

In [ ]:
SAVE_DIR = "./bert_sentiment_final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump({"test_metrics": test_metrics,
               "train_minutes": train_minutes,
               "config": {"model": MODEL_NAME, "max_length": MAX_LENGTH,
                          "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
                          "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "seed": SEED,
                          "config_source": CONFIG_SOURCE}}, f, indent=2)

# Compress and download (EVERYTHING IS DELETED when the Colab session ends!)
!zip -qr bert_sentiment_final.zip {SAVE_DIR}
from google.colab import files
files.download("bert_sentiment_final.zip")
files.download("03_rq1_model_comparison.csv")
files.download("03_bert_confusion_matrix.png")
files.download("03_bert_training_curve.png")

---

## Troubleshooting

| Error | Fix |
|---|---|
| `CUDA out of memory` | Lower `local_batch_size` in `config.py` (e.g. to 2) and raise `local_grad_accum` so their product stays 32 |
| `eval_strategy` unknown | Older transformers: rename `eval_strategy` → `evaluation_strategy` |
| Very slow (hours) | There is no GPU. Runtime → Change runtime type → T4 GPU |
| Colab disconnects | Save the checkpoints to Drive (`output_dir` on Drive) |
| `f1_macro` does not improve after epoch 2 | Normal. 3 epochs are usually enough for BERT. |
| "Settings from: fallback values" | `src/` was not found — upload it next to the notebook, or keep the fallback values in sync with `config.py` |

## Worth trying as well (if there is time)

- **DistilBERT** (`distilbert-base-uncased`): 40% smaller, 60% faster, keeps
  ~97% of the accuracy. A separate row in the table → a strong efficiency argument.
- **RoBERTa** (`roberta-base`): often 1–2 points better than BERT.
- **Domain-specific model**, e.g. `nlptown/bert-base-multilingual-uncased-sentiment`.
- **Cross-dataset test:** run the trained BERT on the TripAdvisor data (RQ5).

**Next:** `04_absa_transformer.ipynb` — HRAST + sentence-pair ABSA (RQ3, RQ4).